# m18 lab · Hybrid and dielectric localization

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

Hybrid systems use a few magnetically tracked points to correct the impedance coordinates of every electrode. Dielectric approaches use the frequency dependence of tissue impedance. You correct a distorted impedance space with an affine map and with a smooth radial-basis correction, see how the error depends on where the reference points are, fuse two estimates by their variances, and compute the impedance phase of two illustrative tissues.

| Part | What you do | Equation |
| --- | --- | --- |
| A | A distorted impedance space | m17 model |
| B | Affine and smooth corrections | eq-m18-affine, eq-m18-rbf |
| C | Error and coverage | eq-m18-coverage |
| D | Fusing two estimates | eq-m18-fusion |
| E | Frequency-dependent impedance | eq-m18-dielectric |

Lengths in mm. The impedance space is the m17 point-patch model with illustrative per-axis scale errors; the chamber is a 45 mm ball; tissue parameters in part E are illustrative, not measured. All data are synthetic. Each part ends with **checks**.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import hybrid as H

## Part A · A distorted impedance space

Impedance coordinates (m17) are smooth but distorted: a uniform-field map stretches toward the patches, and the field
scale is off by a few percent on each axis. Measure the distortion over a chamber-sized ball.

In [ ]:
rng = np.random.default_rng(18)
R = 45.0
g = np.linspace(-R, R, 10)
test = np.stack(np.meshgrid(g, g, g), axis=-1).reshape(-1, 3)
test = test[np.linalg.norm(test, axis=1) <= R]
ut = H.impedance_coordinates(test)
print(f"{len(test)} test points; RMS error of raw impedance coordinates {H.rms(ut - test):.2f} mm, "
      f"max {np.max(np.linalg.norm(ut - test, axis=1)):.2f} mm")

In [ ]:
# Checks
assert H.rms(ut - test) > 2.0

## Part B · Affine and smooth corrections

At points where a magnetic sensor also gives the true position, fit a map from impedance to true coordinates. An affine
map $\mathbf x = A\mathbf u + \mathbf b$ (eq-m18-affine) fixes scale and shear; adding Gaussian radial basis functions
(eq-m18-rbf) also fixes the smooth nonlinearity, if there are enough points.

In [ ]:
rows = []
for n in (10, 30, 100, 300):
    cal = H.ball_points(n, R, rng)
    u = H.impedance_coordinates(cal)
    A, b = H.fit_affine(u, cal)
    m = H.fit_rbf(u, cal, 30.0, 1e-6)
    rows.append((n, H.rms(H.apply_affine(A, b, ut) - test), H.rms(H.apply_rbf(m, ut) - test)))
    print(f"{n:4d} reference points: affine {rows[-1][1]:.3f} mm, smooth {rows[-1][2]:.3f} mm")

In [ ]:
# Checks
assert rows[-1][2] < 0.1 * rows[-1][1]          # with many points the smooth correction wins clearly
assert rows[0][2] > rows[0][1]                  # with very few it overfits: affine is better

## Part C · Error and coverage

A correction is only as good as the reference points near where it is used. Fit with 25 points, then plot each test
point's error against its distance to the nearest reference point (eq-m18-coverage).

In [ ]:
cal = H.ball_points(25, R, rng)
m = H.fit_rbf(H.impedance_coordinates(cal), cal, 30.0, 1e-6)
err = np.linalg.norm(H.apply_rbf(m, ut) - test, axis=1)
dist = H.nearest_distance(test, cal)
plt.scatter(dist, err, s=8); plt.xlabel("distance to the nearest reference point (mm)"); plt.ylabel("error after correction (mm)"); plt.show()
near, far = err[dist < np.median(dist)], err[dist >= np.median(dist)]
print(f"median error: nearer half {np.median(near):.3f} mm, farther half {np.median(far):.3f} mm")

In [ ]:
# Checks
assert np.median(far) > np.median(near)

## Part D · Fusing two estimates

Where an electrode has both a magnetic and a corrected impedance estimate, combine them by inverse variance
(eq-m18-fusion). The combination is never worse than the better one. The SDs are illustrative.

In [ ]:
truth = np.array([10.0, -5.0, 20.0])
n = 20000
mag = truth + 0.5 * rng.standard_normal((n, 3))
imp = truth + 1.5 * rng.standard_normal((n, 3))
fused, var = H.fuse(mag, 0.5**2, imp, 1.5**2)
for name, e in (("magnetic", mag), ("impedance", imp), ("fused", fused)):
    print(f"{name:9s}: SD per axis {np.std(e - truth, axis=0).mean():.3f} mm")
print("predicted fused SD", np.sqrt(var))

In [ ]:
# Checks
assert abs(np.std(fused - truth, axis=0).mean() - np.sqrt(var)) < 0.01

## Part E · Frequency-dependent impedance

Tissue conducts and polarizes: $\sigma^* = \sigma + j\omega\varepsilon_0\varepsilon_r(\omega)$ (eq-m18-dielectric). With one
relaxation, the impedance phase is small at low frequency and largest near the relaxation frequency $1/(2\pi\tau)$.
Two illustrative tissues that differ little at 1 kHz differ clearly near 100 kHz.

In [ ]:
f = np.logspace(2, 7, 300)
for name, p in H.ILLUSTRATIVE_TISSUES.items():
    plt.semilogx(f, H.impedance_phase_deg(f, **p), label=name)
plt.xlabel("frequency (Hz)"); plt.ylabel("impedance phase (degrees)"); plt.legend(); plt.show()
for fi in (1e3, 1e4, 1e5, 1e6):
    pb, pw = (H.impedance_phase_deg(fi, **p) for p in H.ILLUSTRATIVE_TISSUES.values())
    print(f"{fi:9.0f} Hz: phases {pb:6.2f} and {pw:6.2f} degrees")

In [ ]:
# Checks
t = H.ILLUSTRATIVE_TISSUES
d = lambda fi: abs(H.impedance_phase_deg(fi, **t["wall-like"]) - H.impedance_phase_deg(fi, **t["blood-like"]))
assert d(1e3) < 1 and d(1e5) > 10

## Where to go next

- Weight each reference point by its magnetic precision (m16) when fitting the correction.
- Refit the correction over time as the impedance space drifts (m19).
- Replace the single relaxation by a multi-dispersion parametric model and real tissue parameters (Gabriel and
  colleagues).